<!-- d2l:prose id=pooling-md-1 fw=all -->

# Pooling

Convolutional layers answer local questions (whether an edge sits at pixel
$(i, j)$), whereas a classifier must answer a global question about the whole
image. Bridging from local feature maps to a global prediction requires both
wide spatial reach (that section) and robustness to tiny spatial
shifts so a one-pixel camera jitter does not alter the prediction.

A *pooling layer* serves both goals without adding learnable parameters: it
slides a window across each channel independently and summarizes every patch by
its *maximum* or *average*. Because the maximum over a small block stays unchanged
as long as the peak response remains inside that block, local pooling provides
*tolerance* to small shifts, while *global pooling* across the entire feature map
collapses spatial coordinates into a single channel-wise vector.

In [1]:
import math
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

<!-- d2l:prose id=pooling-md-2 fw=all -->

## Maximum Pooling and Average Pooling

Like a convolutional layer, a pooling operator slides a fixed-shape window (the
*pooling window*) across its input and computes a single scalar output at
each position. Unlike a convolution, however, a pooling layer has no
learnable kernel weights: it applies a fixed deterministic reduction to the
elements inside the window. *Maximum pooling* (*max-pooling* for short)
returns the largest entry in the window, whereas *average pooling* returns
their arithmetic mean.

![Max-pooling with a pooling window shape of $2\times 2$. The shaded portions are the first output element as well as the input tensor elements used for the output computation: $\max(0, 1, 3, 4)=4$.](../img/pooling.svg)

In the figure, a $2\times2$ window slides across a $3\times3$ input
with stride 1, moving from left to right and top to bottom. Because the window
fits in two positions along each axis, the output has shape $2\times2$, with
entries given by the maximum of each window:

$$
\begin{aligned}
\max(0, 1, 3, 4) &= 4,\\
\max(1, 2, 4, 5) &= 5,\\
\max(3, 4, 6, 7) &= 7,\\
\max(4, 5, 7, 8) &= 8.
\end{aligned}
$$

Notice that five of the nine input entries (0, 1, 2, 3, and 6) are never the
maximum of any window. Perturbing them slightly leaves the output completely
unchanged, because max-pooling depends only on the winning element in each
window. The same sliding-window construction generalizes immediately to any
$k_\textrm{h} \times k_\textrm{w}$ pooling window.

Below we implement forward propagation for two-dimensional pooling in
`pool2d`. Its nested loop mirrors `corr2d` from that section,
replacing the weighted inner product with either `max` or `mean`. We also
include a `stride` argument so we can examine downsampling next.

In [2]:
def pool2d(X, pool_size, mode='max', stride=1):
    k_h, k_w = pool_size
    Y = d2l.zeros(((X.shape[0] - k_h) // stride + 1,
                   (X.shape[1] - k_w) // stride + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            window = X[i * stride: i * stride + k_h,
                       j * stride: j * stride + k_w]
            Y[i, j] = window.max() if mode == 'max' else window.mean()
    return Y

<!-- d2l:prose id=pooling-md-3 fw=all -->

Testing `pool2d` on the $3\times3$ tensor from the figure confirms
that max-pooling yields 4, 5, 7, and 8.

In [3]:
X = d2l.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
pool2d(X, (2, 2))

tensor([[4., 5.],
        [7., 8.]])

<!-- d2l:prose id=pooling-md-4 fw=all -->

Switching to average pooling on the same input returns the four window means:
2, 3, 5, and 6.

In [4]:
pool2d(X, (2, 2), 'avg')

tensor([[2., 3.],
        [5., 6.]])

<!-- d2l:prose id=pooling-md-5 fw=all -->

## Tolerance to Small Shifts

Just how much spatial shift can pooling absorb? Let us test the simplest
possible setting. A $2\times2$ pooling layer with stride 2 partitions its
input into non-overlapping $2\times2$ blocks and summarizes each block by a
single value. Imagine a $4\times4$ feature map containing a single nonzero
response of 1 (like an edge detector firing at an isolated edge), and slide
that response one pixel at a time across row 1. Below we compare the
max-pooled and average-pooled outputs across three consecutive column
positions.

In [5]:
def single_response(i, j, n=4):
    """An n x n feature map that is 1 at (i, j) and 0 elsewhere."""
    return d2l.tensor([[float(r == i and c == j) for c in range(n)]
                       for r in range(n)])

for j in range(3):
    Z = single_response(1, j)
    print(f'response at (1, {j}): max',
          d2l.numpy(pool2d(Z, (2, 2), stride=2)).tolist(),
          'avg', d2l.numpy(pool2d(Z, (2, 2), 'avg', stride=2)).tolist())

response at (1, 0): max [[1.0, 0.0], [0.0, 0.0]] avg [[0.25, 0.0], [0.0, 0.0]]
response at (1, 1): max [[1.0, 0.0], [0.0, 0.0]] avg [[0.25, 0.0], [0.0, 0.0]]
response at (1, 2): max [[0.0, 1.0], [0.0, 0.0]] avg [[0.0, 0.25], [0.0, 0.0]]


<!-- d2l:prose id=pooling-md-6 fw=all -->

Shifting the response from column 0 to column 1 leaves both pooled outputs
completely unchanged because both locations fall inside the same $2\times2$
block. Shifting it one pixel further to column 2 crosses a block boundary,
causing the pooled response to jump to the adjacent output cell. Thus, a
one-pixel horizontal shift is absorbed whenever the response remains within
its block (half of all starting columns) and shifts the output by one
position otherwise. Local pooling therefore makes a representation *tolerant*
to certain small shifts, not strictly invariant to them.
the figure illustrates the max-pooled outputs across all
four column positions.

![A single response moved through the four columns of a $4\times4$ feature map (top), its $2\times2$ max-pooled map with stride 2 (middle), and its global maximum (bottom). Heavy lines mark the pooling blocks. The pooled response stays put while the response stays inside its block and jumps when it crosses into the next one; the global maximum never moves.](../img/pooling-shift.svg)

Notice also how max-pooling and average pooling summarize the block
differently. Max-pooling preserves the full strength of the activation (1)
anywhere inside the block, effectively asking whether the feature appears
somewhere in that neighborhood. Average pooling outputs 0.25 because the
feature occupies only one of the four cells, measuring the feature's spatial
coverage and diluting a strong isolated response with its zero neighbors.

Subsampling with stride 2 also introduces a subtle signal-processing hazard
whenever a feature crosses a sampling boundary. In the extreme case, taking
every second sample of a high-frequency alternating signal $1, 0, 1, 0, \ldots$
produces either all ones or all zeros depending on a single-pixel shift. This
distortion, occurring whenever a signal varies faster than the sampling rate
can resolve, is called *aliasing*, and it affects every stride-2 downsampler,
whether pooled or convolutional. Applying a small smoothing filter before
subsampling mitigates aliasing [@zhang2019making], a technique explored
in the exercises below.

Historically, shift tolerance was the primary motivation behind pooling.
Fukushima's Neocognitron [@Fukushima.1980] paired each layer of
feature-detecting cells with pooling-like cells that fired whenever a feature
appeared anywhere in a local neighborhood. An early version of max-pooling
appeared in speech recognition [@Yamaguchi.Sakamoto.Akabane.ea.1990], and
max-pooling was subsequently proposed in computational neuroscience as a model
of hierarchical evidence aggregation in the visual cortex
[@Riesenhuber.Poggio.1999].

## Global Pooling

What about the full translation invariance we desired in that section
when asking whether Waldo appears *anywhere* in an image? As we just saw, a
small fixed window cannot guarantee it. Expanding the pooling window to cover
the *entire* spatial extent of each feature map can: the maximum or mean over
all spatial positions yields the exact same scalar regardless of where the
response sits (bottom row of the figure). This operation is
called *global pooling*, and it collapses each channel to a single number.


Inside a full CNN, however, global pooling sits atop zero-padded convolutional
layers, which are strictly shift-equivariant only away from the image borders
(that section). As we saw in that section, an
interior pixel participates in all nine windows of a $3 \times 3$ kernel whereas
a border pixel participates in only six (and a corner pixel in four), so while a
global maximum over an isolated response stays constant across columns, a global
average drops when a feature moves onto the padded border. Stride-1
convolutional stacks ending in global pooling are therefore strictly invariant
to shifts that keep salient content away from the boundaries, and approximately
invariant otherwise. Strided layers weaken this guarantee: because of the
aliasing we met above, the invariance is then exact only for shifts by a
multiple of the product of the strides.

In the modern architectures of that section, the two roles of
pooling are frequently separated. Intermediate spatial downsampling is often
handled by stride-2 convolutions, which learn how to weight each window rather
than committing to a fixed maximum or average, while the network terminates in
*global average pooling* [@Lin.Chen.Yan.2013] before the final linear
classifier. LeNet, the classic architecture we build in the next section,
predates both conventions.

## Padding and Stride

Because pooling slides a window over spatial grids in exactly the same way as
convolution, deep learning frameworks parameterize pooling layers with the
same window shape, padding, and stride arguments, and the output shape obeys
the identical formula from that section: padding $p$ pixels on each
side of an axis of length $n$ with window size $k$ and stride $s$ yields
$\lfloor(n+2p-k)/s\rfloor+1$ outputs along that axis. The one subtle
difference is what *value* fills the padded border. To demonstrate these
behaviors with built-in pooling layers, we first construct a four-dimensional
tensor holding a single $4\times4$ feature map with entries 0 through 15.

In [6]:
# (example, channel, height, width)
X = d2l.reshape(d2l.arange(16, dtype=d2l.float32), (1, 1, 4, 4))
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]]]])

<!-- d2l:prose id=pooling-md-7 fw=all -->

Because pooling is usually used to summarize non-overlapping regions, most
frameworks default to setting the stride equal to the pooling window size so
the windows tile the input without overlap. On our $4\times4$ input, a
$3\times3$ window with stride 3 fits only once, returning the maximum of the
upper-left $3\times3$ block (10). Since pooling has no learnable weights, the
layer requires no parameter initialization.

In [7]:
pool = nn.MaxPool2d(3)  # The stride defaults to the window shape
pool(X)

tensor([[[[10.]]]])

<!-- d2l:prose id=pooling-md-8 fw=all -->

We can override these defaults whenever we want overlapping windows or
explicit padding. Setting padding to 1 and stride to 2 produces an output of
size $\lfloor(4+2-3)/2\rfloor+1 = 2$ along both height and width.

In [8]:
pool = nn.MaxPool2d(3, padding=1, stride=2)
pool(X)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])

<!-- d2l:prose id=pooling-md-9 fw=all -->

What value should fill the padded border of a pooling layer? In a convolution,
zero is the natural neutral element because multiplying by zero contributes
nothing to a weighted sum. For max-pooling, however, the neutral element is
$-\infty$: padded cells can never beat a real activation, ensuring every
output equals the maximum over the genuine input entries inside its window. In
our example above, the windows cover real rows and columns $\{0, 1\}$ and
$\{1, 2, 3\}$, yielding 5, 7, 13, and 15. Padding with $-\infty$ also
preserves shift equivariance in *value*: adding a constant to every input
entry shifts every pooled output by the exact same constant. We can verify
this on `X - 16`, whose entries are all strictly negative.

In [9]:
pool(X - 16)

tensor([[[[-11.,  -9.],
          [ -3.,  -1.]]]])

<!-- d2l:prose id=pooling-md-10 fw=all -->

Every output decreases by exactly 16, as expected. Had the layer padded with
zeros instead, the three windows touching the border would have returned 0,
corrupting the result simply because 0 happens to be larger than all negative
inputs.

Average pooling has no analogous neutral constant. Including padded zeros in
the denominator biases border averages toward 0; the cleaner alternative is to
divide only by the count of real input entries falling inside each window.
<!-- d2l:prose id=pooling-md-11 fw=mxnet, pytorch, jax -->

Average pooling here counts the padded zeros by default
(`count_include_pad=True`); `count_include_pad=False` averages only the real
entries.

<!-- d2l:prose id=pooling-md-12 fw=all -->

## Multiple Channels

Whereas a convolutional layer mixes information across all input channels by
summing over them (that section), a pooling layer operates
independently on each channel. Consequently, the output always has the exact
same number of channels as the input. To verify this behavior, we concatenate
`X` and `X + 1` along the channel axis to create a two-channel tensor.

In [10]:
X = d2l.concat((X, X + 1), 1)
X

tensor([[[[ 0.,  1.,  2.,  3.],
          [ 4.,  5.,  6.,  7.],
          [ 8.,  9., 10., 11.],
          [12., 13., 14., 15.]],

         [[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])

<!-- d2l:prose id=pooling-md-13 fw=all -->

Applying our earlier pooling operation (padding 1, stride 2) yields two output
channels: the first reproduces 5, 7, 13, and 15, while the second returns the
same values incremented by 1.

In [11]:
pool(X)

tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])

<!-- d2l:prose id=pooling-md-14 fw=all -->

Because pooling has no learnable weights, it discovers no patterns on its
own; it can only summarize the features detected by preceding convolutional
layers. With convolutions, padding, stride, dilation, multi-channel feature
maps, and pooling all in hand, we now have every building block needed to
assemble and train a complete convolutional neural network and compare it
directly against our earlier MLPs.

## Summary

Pooling layers summarize each spatial window of an input feature map by its
maximum or average. They introduce zero learnable parameters, process each
channel independently, and follow the exact same padding and stride output-shape
rules as convolutions; max-pooling pads borders with $-\infty$ so that padded
positions never affect the maximum. A $2\times2$ pooling layer with stride 2
halves spatial resolution along each axis and absorbs small shifts that stay
within a block, while still jumping when a response crosses a block boundary.
Max-pooling preserves the peak activation in a neighborhood, whereas average
pooling dilutes isolated peaks across the window. Finally, global pooling
collapses each entire feature map to a single scalar. On top of stride-1
layers this yields full translation invariance for interior patterns (and
approximate invariance near padded borders); with strided layers the
invariance is exact only for shifts by a multiple of the product of the
strides.

## Exercises

1. **Pooling alternatives.**
    1. Do we need a separate minimum-pooling layer, or can it be expressed
       through an operation already in this section?
    1. *Softmax pooling* with temperature $\tau > 0$ replaces the window
       entries $x_1, \ldots, x_m$ by their softmax-weighted average
       $$\sum_{i=1}^m x_i \frac{\exp(x_i / \tau)}{\sum_{j=1}^m \exp(x_j / \tau)}.$$
       Find its limits as $\tau \to 0$ and as $\tau \to \infty$. Give one
       cost that softmax pooling adds relative to max-pooling.

2. **Aliasing and blur-pool.** Naive stride-2 downsampling keeps
   every second entry of its input. Measure frequency in cycles per input
   sample, so that the one-dimensional signals $(1, 0, 1, 0, 1, 0)$ and
   $(0, 1, 0, 1, 0, 1)$ both have frequency $\tfrac{1}{2}$.
    1. Apply naive stride-2 downsampling to both signals, which differ by
       a shift of one position, and compare the two outputs. Why is this
       behavior called *aliasing*?
    1. Which input frequencies can a stride-2 subsampler represent
       faithfully?
    1. Blur-pool [@zhang2019making] counteracts aliasing by applying
       a small averaging filter before subsampling. Work out what
       blur-pool with a two-tap box filter
       $(\tfrac{1}{2}, \tfrac{1}{2})$ computes on the two signals above.
       Which pooling operation from this section does it coincide with?

3. [code] **Backward pass of max-pooling.** Derive the gradient of
   two-dimensional max-pooling with respect to its input, for a
   $k_\textrm{h} \times k_\textrm{w}$ window and stride equal to the
   window size, and implement it without automatic differentiation.
   Verify against the gradient that automatic differentiation computes
   through the built-in max-pooling operation of this section on a small
   random tensor, to a relative error below $10^{-4}$.

    *Adapted from Stanford CS231n,
    [Assignment 2](https://cs231n.github.io/assignments2024/assignment2/),
    ConvolutionalNetworks, naive max-pooling backward pass.*

4. **Max-pooling via ReLU.** ● A convolution is linear, so it cannot
   compute a maximum by itself.
    1. Express $\max(a, b)$ using only affine operations and
       $\textrm{ReLU}(x) = \max(0, x)$.
    1. Use this to implement max-pooling as a stack of convolutions and
       ReLU layers.
    1. Count the channels per layer and the ReLU layers that your
       construction uses for a $2 \times 2$ window and for a
       $3 \times 3$ window. How does the number of ReLU layers grow with
       the number of entries in the window?

<!-- d2l:prose id=pooling-md-15 fw=pytorch -->

[Discussions](https://d2l.discourse.group/t/72)